In [ ]:
import mysql.connector

# Connect to MySQL
con = mysql.connector.connect(
    host="localhost",
    user="root",
    password="10032004",
    database="company_db"
)


# Check if employee exists
def check_employee(Id):
    sql = "SELECT * FROM employee WHERE id = %s"

    cursor = con.cursor(buffered=True)
    data = (Id,)

    cursor.execute(sql, data)
    employee = cursor.fetchone()

    cursor.close()

    return employee is not None


# Add Employee
def add_employee():

    Name = input("Enter Employee Name: ")
    Post = input("Enter Employee Post: ")

    try:
        Salary = float(input("Enter Employee Salary: "))
    except ValueError:
        print("Invalid salary. Please enter a number.")
        return

    sql = """
        INSERT INTO employee (name, position, salary, status)
        VALUES (%s, %s, %s, %s)
    """

    data = (Name, Post, Salary, "Active")

    cursor = con.cursor()

    try:
        cursor.execute(sql, data)
        con.commit()

        # Get automatically generated employee ID
        employee_id = cursor.lastrowid

        print("\nEmployee Added Successfully")
        print("Employee ID:", employee_id)
        print("Status: Active")

    except mysql.connector.Error as err:
        print(f"Error: {err}")
        con.rollback()

    finally:
        cursor.close()


# Mark Employee as Inactive
def mark_employee_inactive():

    Id = input("Enter Employee ID: ")

    if not check_employee(Id):
        print("Employee Does Not Exist. Please Try Again.")
        return

    sql = """
        UPDATE employee
        SET status = 'Inactive'
        WHERE id = %s
    """

    data = (Id,)
    cursor = con.cursor()

    try:
        cursor.execute(sql, data)
        con.commit()

        if cursor.rowcount > 0:
            print("Employee Status Changed to Inactive Successfully")

    except mysql.connector.Error as err:
        print(f"Error: {err}")
        con.rollback()

    finally:
        cursor.close()


# Mark Employee as Active
def mark_employee_active():

    Id = input("Enter Employee ID: ")

    if not check_employee(Id):
        print("Employee Does Not Exist. Please Try Again.")
        return

    sql = """
        UPDATE employee
        SET status = 'Active'
        WHERE id = %s
    """

    data = (Id,)
    cursor = con.cursor()

    try:
        cursor.execute(sql, data)
        con.commit()

        if cursor.rowcount > 0:
            print("Employee Status Changed to Active Successfully")

    except mysql.connector.Error as err:
        print(f"Error: {err}")
        con.rollback()

    finally:
        cursor.close()


# Promote Employee
def promote_employee():

    Id = input("Enter Employee's ID: ")

    if not check_employee(Id):
        print("Employee Does Not Exist. Please Try Again.")
        return

    cursor = con.cursor()

    try:
        # Check employee status
        sql_status = """
            SELECT salary, status
            FROM employee
            WHERE id = %s
        """

        data = (Id,)
        cursor.execute(sql_status, data)

        result = cursor.fetchone()

        if result is None:
            print("Employee does not exist.")
            return

        current_salary = float(result[0])
        status = result[1]

        # Don't promote inactive employees
        if status == "Inactive":
            print("Cannot promote an inactive employee.")
            return

        try:
            Amount = float(input("Enter Increase in Salary: "))
        except ValueError:
            print("Invalid amount. Please enter a number.")
            return

        new_salary = current_salary + Amount

        # Update salary
        sql_update = """
            UPDATE employee
            SET salary = %s
            WHERE id = %s
        """

        data_update = (new_salary, Id)

        cursor.execute(sql_update, data_update)

        con.commit()

        print("\nEmployee Promoted Successfully")
        print("Old Salary :", current_salary)
        print("New Salary :", new_salary)

    except mysql.connector.Error as err:
        print(f"Error: {err}")
        con.rollback()

    finally:
        cursor.close()


# Display Employees
def display_employee():

    cursor = None

    try:
        sql = """
            SELECT id, name, position, salary, status
            FROM employee
            ORDER BY id
        """

        cursor = con.cursor()

        cursor.execute(sql)

        employees = cursor.fetchall()

        if not employees:
            print("No employees found.")
            return

        print("\n========== EMPLOYEE RECORDS ==========")

        for employee in employees:

            print("Employee ID     :", employee[0])
            print("Employee Name   :", employee[1])
            print("Employee Post   :", employee[2])
            print("Employee Salary :", employee[3])
            print("Employee Status :", employee[4])

            print("------------------------------------")

    except mysql.connector.Error as err:
        print(f"Error: {err}")

    finally:
        if cursor:
            cursor.close()

# Edit Employee Data
def edit_employee():

    Id = input("Enter Employee ID: ")

    if not check_employee(Id):
        print("Employee Does Not Exist. Please Try Again.")
        return

    print("\nWhat do you want to edit?")
    print("1. Employee Name")
    print("2. Employee Position")
    print("3. Employee Salary")

    choice = input("Enter Your Choice: ")

    cursor = con.cursor()

    try:

        if choice == "1":
            new_name = input("Enter New Employee Name: ")

            sql = "UPDATE employee SET name = %s WHERE id = %s"
            cursor.execute(sql, (new_name, Id))

        elif choice == "2":
            new_position = input("Enter New Employee Position: ")

            sql = "UPDATE employee SET position = %s WHERE id = %s"
            cursor.execute(sql, (new_position, Id))

        elif choice == "3":
            try:
                new_salary = float(input("Enter New Employee Salary: "))
            except ValueError:
                print("Invalid salary. Please enter a number.")
                return

            sql = "UPDATE employee SET salary = %s WHERE id = %s"
            cursor.execute(sql, (new_salary, Id))

        else:
            print("Invalid Choice!")
            return

        con.commit()
        print("Employee Data Updated Successfully")

    except mysql.connector.Error as err:
        print(f"Error: {err}")
        con.rollback()

    finally:
        cursor.close()


# Menu
def menu():

    while True:

        print("\nWelcome to Employee Management Record")
        print("------------------------------------")
        print("1. Add Employee")
        print("2. Edit Employee Details")
        print("3. Mark Employee Inactive")
        print("4. Mark Employee Active")
        print("5. Promote Employee")
        print("6. Display Employees")
        print("7. Exit")

        ch = input("Enter Your Choice: ")

        if ch == "1":
            add_employee()

        elif ch == '2':
            edit_employee()

        elif ch == "3":
            mark_employee_inactive()

        elif ch == "4":
            mark_employee_active()

        elif ch == "5":
            promote_employee()

        elif ch == "6":
            display_employee()

        elif ch == "7":
            print("Exiting the Program. Goodbye!")
            break

        else:
            print("Invalid Choice! Please Try Again.")


# Start program
if __name__ == "__main__":
    menu()


Welcome to Employee Management Record
------------------------------------
1. Add Employee
2. Edit Employee Details
3. Mark Employee Inactive
4. Mark Employee Active
5. Promote Employee
6. Display Employees
7. Exit


Enter Your Choice:  1
Enter Employee Name:  Rick
Enter Employee Post:  Engineer
Enter Employee Salary:  20000



Employee Added Successfully
Employee ID: 9
Status: Active

Welcome to Employee Management Record
------------------------------------
1. Add Employee
2. Edit Employee Details
3. Mark Employee Inactive
4. Mark Employee Active
5. Promote Employee
6. Display Employees
7. Exit


Enter Your Choice:  2
Enter Employee ID:  9



What do you want to edit?
1. Employee Name
2. Employee Position
3. Employee Salary


...

Employee Status Changed to Inactive Successfully

Welcome to Employee Management Record
------------------------------------
1. Add Employee
2. Edit Employee Details
3. Mark Employee Inactive
4. Mark Employee Active
5. Promote Employee
6. Display Employees
7. Exit


Enter Your Choice:  4
Enter Employee ID:  9


Employee Status Changed to Active Successfully

Welcome to Employee Management Record
------------------------------------
1. Add Employee
2. Edit Employee Details
3. Mark Employee Inactive
4. Mark Employee Active
5. Promote Employee
6. Display Employees
7. Exit


Enter Your Choice:  6



========== EMPLOYEE RECORDS ==========
Employee ID     : 1
Employee Name   : Aadil
Employee Post   : Manager
Employee Salary : 70000
Employee Status : Active
------------------------------------
Employee ID     : 2
Employee Name   : Afraz
Employee Post   : Data Analyst
Employee Salary : 40000
Employee Status : Inactive
------------------------------------
Employee ID     : 3
Employee Name   : Rayaan
Employee Post   : Receptionist
Employee Salary : 30000
Employee Status : Active
------------------------------------
Employee ID     : 4
Employee Name   : Kaushik
Employee Post   : Electrician
Employee Salary : 20000
Employee Status : Active
------------------------------------
Employee ID     : 5
Employee Name   : Afnan
Employee Post   : Accountant
Employee Salary : 50000
Employee Status : Active
------------------------------------
Employee ID     : 6
Employee Name   : Afzal
Employee Post   : Employer
Employee Salary : 60000
Employee Status : Active
------------------------------------
E